# Dalitz fit: $\eta \to \mu^+\mu^-\gamma$

The workflow is similar to the mass fit, except organized by bins in the dalitz plot:

    read ROOT -> cuts -> random candidate -> get dalitz variables   driver_dalitz.py
              -> global fit_dcb -> per-dalitz-bin fits              fit_dalitz.fit_dalitz
              -> plots                                              plot_dalitz_bins.plot_dalitz_fit

This notebook runs the selection, the fit and its checks on one ROOT file and
one factor. The command-line equivalent is `driver_dalitz.py`.

In [ ]:
import json
import os
import sys
from datetime import date

import numpy as np
from IPython.display import Image, display

# Notebooks don't define __file__; fall back to cwd.
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
for sub in ('utils', 'fit', 'plot'):
    sys.path.insert(0, os.path.join(_here, '..', sub))

import fit_utils
import kinematics
from driver_dalitz import select_candidates
from fit_dalitz import fit_dalitz, hole_neighbors
from fit_dcb import TAIL_PARAMS
from plot_dalitz import plot_dalitz_hist
from plot_dalitz_bins import plot_dalitz_fit

# plot_phase_space (imported by the plot modules) selects the Agg backend
%matplotlib inline

In [ ]:
# Configuration
# Input ROOT file
INFILE = '../../ntuple/lhcb/eta2mumugamma/20261006/combined_files.root'
OUTDIR = f'../../out/eta2mumugamma/turbo/{date.today():%Y%m%d}/dalitz/notebook/'

# Global fit's DCB tail prior: the data DCB fit of run_mass_fits.sh, the TAIL_PRIOR of
# run_dalitz.sh, widened by TAIL_PRIOR_FACTOR standard deviations. None = no prior.
TAIL_PRIOR_PATH = '../../out/eta2mumugamma/turbo/20261105/mc_prior/results_dcb.json'
TAIL_PRIOR_FACTOR = 5.0

# Mass fit window. MASS_NBINS is the number of bins in the mass histogram per
# Dalitz bin.
XMIN, XMAX, MASS_NBINS = 450.0, 650.0, 200
MEAN_INIT, SIGMA_INIT = kinematics.M_ETA, 10.0

# Dalitz binning: N_PER_AXIS x N_PER_AXIS bins over M12_RANGE x M23_RANGE [GeV^2]
N_PER_AXIS = 14
M12_RANGE = (0.0, 0.21)
M23_RANGE = (0.0, 0.31)
MIN_CANDIDATES = 10    # this many candidates in a Dalitz bin or fewer -> don't attempt a fit

# Per-bin fit prior scaling factor on mean/sigma. The larger the factor, the
# looser the prior uncertainty.
SIGNAL_PRIOR_FACTOR = 5

# Per-bin factor f(M) on the signal and the background shape: None (plain
# shapes), kinematics.phase_space_factor, or kinematics.dalitz_factor (signal).
SIGNAL_FACTOR = kinematics.phase_space_factor
BKG_FACTOR = kinematics.phase_space_factor

os.makedirs(OUTDIR, exist_ok=True)

## Read and select

`driver_dalitz.select_candidates` reads the branches, drops events that can't
be interpreted, applies the TOS and muon cuts, reduces each event to one
randomly picked candidate, and computes the Dalitz
variables and the mass from the candidate's daughters. Its cutflow is printed
and written to `selection.json`.

In [ ]:
m12s, m23s, masses, selection = select_candidates(INFILE)
with open(os.path.join(OUTDIR, 'selection.json'), 'w') as f:
    json.dump(selection, f, indent=2)

## Raw Dalitz distributions

Create dalitz plot without any fitting. Generate plots for both the full reconstructed range and zoomed into the physical region.

In [ ]:
for name, hist_range in [('dalitz_raw.png', [[0, 1.0], [0, 1.2]]),
                         ('dalitz_raw_zoom.png', [[0, 0.22], [0, 0.35]])]:
    plot_dalitz_hist(m12s, m23s, hist_range, os.path.join(OUTDIR, name))
    display(Image(os.path.join(OUTDIR, name), width=600))

## Global and per-Dalitz-bin fits

`fit_dalitz` fits every candidate with `fit_dcb`, then each Dalitz bin. The
global fit seeds the bins and constrains their mean and sigma; the tails are
fixed at its values.

In [ ]:
tail_prior = None
if TAIL_PRIOR_PATH is not None:
    with open(TAIL_PRIOR_PATH) as f:
        mc_fit = json.load(f)['fit']
    tail_prior = fit_utils.build_prior_from_result(mc_fit, TAIL_PARAMS, factor=TAIL_PRIOR_FACTOR)

m12_edges = np.linspace(*M12_RANGE, N_PER_AXIS + 1)
m23_edges = np.linspace(*M23_RANGE, N_PER_AXIS + 1)
result, m_global = fit_dalitz(m12s, m23s, masses, m12_edges, m23_edges,
                              xmin=XMIN, xmax=XMAX, nbins=MASS_NBINS,
                              mean_init=MEAN_INIT, sigma_init=SIGMA_INIT,
                              tail_prior=tail_prior, signal_prior_factor=SIGNAL_PRIOR_FACTOR,
                              min_candidates=MIN_CANDIDATES,
                              signal_factor=SIGNAL_FACTOR, bkg_factor=BKG_FACTOR)
result['meta']['input_file'] = INFILE
result['meta']['tail_prior'] = (None if TAIL_PRIOR_PATH is None else
                                {'path': TAIL_PRIOR_PATH, 'factor': TAIL_PRIOR_FACTOR})
dalitz = result['dalitz']

In [ ]:
# Debug: HESSE diagnostics per parameter of the global fit.
print(m_global.fmin)
for p in m_global.parameters:
    print(f'  {p:8s} value={m_global.values[p]:12.4f}  '
          f'error={m_global.errors[p]:10.4f}  limits={m_global.limits[p]}')

## Fit sanity checks

In [ ]:
# A hole (yield = 0) is either too few candidates to attempt a fit, or a fit
# that ran and failed to converge. If a hole's n_cand is comparable to its
# neighbors', the 0 in the yield map is a fit-convergence issue.
outcome = dalitz['outcome']
print(f'--- Per-bin fit outcome ({N_PER_AXIS}x{N_PER_AXIS} = {N_PER_AXIS**2} Dalitz bins) ---')
print(f"  skipped (<= {MIN_CANDIDATES} candidates): {outcome['skipped']}")
print(f"  fit invalid (hole):               {outcome['invalid']}")
print(f"  fit valid:                        {outcome['valid']}")
for i, j, n_cand, neighbors in hole_neighbors(result):
    neighbor_text = ', '.join(f'{offset}: {count:.0f}' for offset, count in neighbors.items())
    print(f'  hole ({i:2d},{j:2d}): n_cand={n_cand:.0f}  neighbors: {neighbor_text}')

In [ ]:
# Background positivity: valid fits whose Chebyshev, before the BKG_FLOOR
# floor, goes below 0 at a mass the background can reach.
valid = [b for b in dalitz['bins'] if b['valid']]
negative = [b for b in valid if b['bkg_goes_negative']]
print(f'--- Background positivity check: {len(negative)} / {len(valid)} valid fits negative ---')
for b in negative:
    print(f"  ({b['i']:2d},{b['j']:2d}): "
          + '  '.join(f'{name}={b[name]:.3f}' for name in fit_utils.BKG_PARAMS))

In [ ]:
# Per-bin yields should sum to roughly the global fit's numbers, short by the
# out-of-grid fraction.
closure = dalitz['yield_closure']
print('--- Global vs summed per-bin yields ---')
print(f"  window candidates in the bins : {closure['window_candidates_in_bins']} / "
      f"{closure['window_candidates']}")
print(f"  n_sig  global {closure['n_sig_global']:10.1f}   summed {closure['n_sig_summed']:10.1f}")
print(f"  n_bkg  global {closure['n_bkg_global']:10.1f}   summed {closure['n_bkg_summed']:10.1f}")
chi2 = dalitz['chi2_per_ndof_over_bins']
if chi2 is not None:
    print(f"  chi2/ndof over valid bins: mean {chi2['mean']:.3f}, median {chi2['median']:.3f}, "
          f"max {chi2['max']:.3f}")

## Results file and figures

The global fit dict plus a `dalitz` section recording the per-bin outcome,
then the global fit, the signal and background yield maps, and one page per
Dalitz bin (including bins that were skipped or didn't converge).

In [ ]:
factor_label = ('none' if SIGNAL_FACTOR is None else
                'weighted' if SIGNAL_FACTOR is kinematics.dalitz_factor else 'flat')
json_path = os.path.join(OUTDIR, f'dalitz_fit_{factor_label}.json')
with open(json_path, 'w') as f:
    json.dump(result, f, indent=1)
print(f'[DONE] {json_path}')

prefix = os.path.splitext(json_path)[0]
plot_dalitz_fit(result, prefix)
for name in ('global_fit.png', f'dalitz_fit_{factor_label}_signal_yield.png',
             f'dalitz_fit_{factor_label}_bkg_yield.png'):
    display(Image(os.path.join(OUTDIR, name), width=600))